## EMOBON Questions

This file was made to answer specific questions about emobon to see its capabilities for data harvesting and data management.
The questions posed originated from https://vliz.atlassian.net/wiki/spaces/VMDCOS/pages/204374034/First+set+of+queries+to+run+on+3store+of+emo+bon+data

The top level questions are as follows:

- 1: Give me the list of observatories: their names, country, location (lat, long and mrgid), and any habitat info
- 2: For each observatory now, give me the names, the number of sampling events for water and sediment and the overall date coverage, and the number of samples taken
- 3: For a named observatory (so I input the name), give me the number and percentage of missing information in the mandatory fields, per event (not per sample!). NA is “missing information, by the way so don’t throw them out. Plot or tabulate this per mandatory field name (as given in the logsheets)
- 4: And also the summary of the %coverage in the mandatory fields for the sampling tab (that is, the sampling tab of the googlesheet) and of the measured tab separately. do not throw out the NAs here.
- 5: For the mandatory measurements, give me the mean and mean deviation, as well as max and min values, for a named observatory (so I input the name), summed over all sampling events and over all water events separately (plots could be side-by-side). Give me this following the BODC names but then also output a list of BODC vs googlesheet names so I can read that off. Throw out the NAs here
- 6: Over all the observatories, list the samp_collect_devices used and the number of events (not samples) each was used for (I want to see how many different ones there are). Here you can throw out the NAs before reporting.
--> matched to query e (collect_device_usage)
- 7: For each observatory, how many samples for water and for sediment separately have been shipped to HQ
- 8: Identify the metagenomic sampling events taken in the English Channel (aka La Manche) during 2022-2023 where the sea temperature was below 10 degrees Celsius and the abundance of the taxon E. coli was > 2% (all of this is within the EMO BON data sets)

For each of these questions also make a query template that can be used for an anduser to have some variables that can be used in UDAL.

In [18]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [19]:
# import all stuff needed
from conneg_functions import execute_to_df, generate_sparql

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from IPython.core.display import HTML
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output
from sema.query import DefaultSparqlBuilder, GraphSource as KGSource, QueryResult
from pathlib import Path
import os
import re
from pandas import DataFrame

In [20]:
# some paramter setup for the triplestore
# SPARQL EndPoint to use - wrapped as Knowledge-Graph 'source'
GDB_BASE: str = os.getenv("GDB_BASE", "http://graphdb:7200/")
# print(f"{os.getenv('GDB_BASE')=}")
# print(f"{GDB_BASE=}")
GDB_REPO: str = os.getenv("GDB_REPO", "kgap")
GDB_ENDPOINT: str = f"{GDB_BASE}repositories/{GDB_REPO}"
# print(f"{GDB_ENDPOINT=}")
GDB: KGSource = KGSource.build(GDB_ENDPOINT)

# some css to format tables so that they are readable
HTML("""
<style>
    .dataframe td, .dataframe th {
        min-width: 500px;
        word-wrap: break-word;
    }
</style>
""")

In [21]:
# - 1: Give me the list of observatories: their names, country, location (lat, long and mrgid), and any habitat info

sparql_observatory = '''
PREFIX owl: <http://www.w3.org/2002/07/owl#> 
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX emobon: <https://data.emobon.embrc.eu/ns/core#>
SELECT DISTINCT ?observatory 
(GROUP_CONCAT(DISTINCT ?marine_region; SEPARATOR="|") AS ?marine_regions)
(GROUP_CONCAT(DISTINCT ?marine_region_id; SEPARATOR="|") AS ?marine_region_ids)
(GROUP_CONCAT(DISTINCT ?country; SEPARATOR="|") AS ?countries)
(GROUP_CONCAT(DISTINCT ?biome; SEPARATOR="|") AS ?biomes)
WHERE  {
    ?observatory a emobon:Observatory .
    OPTIONAL { ?observatory emobon:originCountry ?country . }
    OPTIONAL { ?observatory emobon:marineRegionName ?marine_region . }
    OPTIONAL { ?observatory emobon:marineRegion ?marine_region_id . }
    OPTIONAL { ?observatory emobon:broadBiome ?biome . }
}
GROUP BY ?observatory
'''

result: QueryResult = GDB.query(sparql=sparql_observatory)
print(f"Result: {result=}")
df_observatory: DataFrame = result.to_dataframe()

HTML("""
<style>
    .dataframe td, .dataframe th {
        word-wrap: break-word;
    }
</style>
""" + df_observatory.to_html())

df_observatory


Result: result=<sema.query.query.SPARQLQueryResult object at 0x7f695af91590>


,observatory,marine_regions,marine_region_ids,countries,biomes
0,http://data.emobon.embrc.eu/observatory-hcmr-1...,Crete Sea|Mediterranean Sea - Eastern Basin|Ae...,http://marineregions.org/mrgid/3315|http://mar...,Greece,marine%20biome%20%5BENVO:00000447%5D|marine%20...
1,http://data.emobon.embrc.eu/observatory-mbal4-...,Western Channel|English Channel|North Atlantic...,http://marineregions.org/mrgid/17527|http://ma...,United Kingdom,marine%20biome%20%5BENVO:00000447%5D|marine%20...
2,http://data.emobon.embrc.eu/observatory-iuieil...,Indian Ocean|Gulf of Eilat,http://marineregions.org/mrgid/4263|http://mar...,Israel,marine%20biome%20%5BENVO:00000447%5D|marine%20...
3,http://data.emobon.embrc.eu/observatory-bpns-c...,North Atlantic Ocean|North Sea|Belgian part of...,http://marineregions.org/mrgid/1912|http://mar...,Belgium,marine%20biome%20%5BENVO:00000447%5D|marine%20...
4,http://data.emobon.embrc.eu/observatory-bpns-c...,North Atlantic Ocean|North Sea|Belgian part of...,http://marineregions.org/mrgid/1912|http://mar...,Belgium,marine%20biome%20%5BENVO:00000447%5D|marine%20...
5,http://data.emobon.embrc.eu/observatory-esc68n...,Norwegian Sea|Arctic Ocean|Norwegian part of t...,http://marineregions.org/mrgid/1906|http://mar...,Norway,marine%20biome%20%5BENVO:00000447%5D|marine%20...
6,http://data.emobon.embrc.eu/observatory-roskog...,English Channel|North Atlantic Ocean|French pa...,http://marineregions.org/mrgid/2389|http://mar...,France,marine%20biome%20%5BENVO:00000447%5D|marine%20...
7,http://data.emobon.embrc.eu/observatory-roskog...,English Channel|North Atlantic Ocean|French pa...,http://marineregions.org/mrgid/2389|http://mar...,France,marine%20biome%20%5BENVO:00000447%5D|marine%20...
8,http://data.emobon.embrc.eu/observatory-rformo...,North Atlantic Ocean|Ria Formosa|Atlantic Ocean,http://marineregions.org/mrgid/1912|http://mar...,Portugal,marine%20biome%20%5BENVO:00000447%5D
9,http://data.emobon.embrc.eu/observatory-rformo...,North Atlantic Ocean|Ria Formosa|Atlantic Ocean,http://marineregions.org/mrgid/1912|http://mar...,Portugal,marine%20biome%20%5BENVO:00000447%5D|marine%20...


In [35]:
# - 2: For each observatory now, give me the names, the number of sampling events for water and sediment and the overall date coverage, and the number of samples taken

sparql_observatory_observations = '''
PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
select * where { 
	?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    #?s sosa:hasFeatureOfInterest ?foi .
    #?foi sosa:isResultOf ?se .
    #?se smp:linkedToObservatory ?obs .
}
'''

# result: QueryResult = GDB.query(sparql=sparql_observatory_observations)
# df_sparql_observatory_observations: DataFrame = result.to_dataframe()

# now we add the hasMixsPackage to the query to get the sample type
sparql_observatory_observations = '''
PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX cvoc: <https://data.emobon.embrc.eu/ns/core_vocab#>
select * where { 
	?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    ?obs core:sampleType ?sample_type .
    #?s sosa:hasFeatureOfInterest ?foi .
    #?foi sosa:isResultOf ?se .
    #?se smp:linkedToObservatory ?obs .
}
'''

# result: QueryResult = GDB.query(sparql=sparql_observatory_observations)
# df_sparql_observatory_observations: DataFrame = result.to_dataframe()

# Modify the query to group by observatory and count the number of observations per observatory
sparql_observatory_observations_grouped = '''
PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX cvoc: <https://data.emobon.embrc.eu/ns/core_vocab#>
SELECT ?obs_id ?obs ?sample_type  (COUNT(?s) AS ?observation_count)
WHERE { 
    ?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    ?obs core:observatoryId ?obs_id .
    ?obs core:sampleType ?sample_type .
}
GROUP BY ?obs ?sample_type ?obs_id
'''

# Execute the modified query
# result_grouped: QueryResult = GDB.query(sparql=sparql_observatory_observations_grouped)
# df_grouped_observations: DataFrame = result_grouped.to_dataframe()
# df_grouped_observations

# now we add the date coverage to the query to get the sample type
sparql_observatory_observations_grouped_and_dates = '''
PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX cvoc: <https://data.emobon.embrc.eu/ns/core_vocab#>
SELECT ?obs_id ?obs ?se ?start_time ?end_time ?sample_type  (COUNT(?s) AS ?observation_count)
WHERE { 
    ?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf ?se .
    ?obs core:observatoryId ?obs_id .
    ?obs core:sampleType ?sample_type .
    ?se <http://www.w3.org/ns/prov#startedAtTime> ?start_time .
    ?se sosa:resultTime ?end_time .
}
GROUP BY ?obs ?sample_type ?obs_id ?se ?start_time ?end_time
'''

# Execute the modified query
# result_grouped_and_dates: QueryResult = GDB.query(sparql=sparql_observatory_observations_grouped_and_dates)
# df_grouped_and_dates: DataFrame = result_grouped_and_dates.to_dataframe()
# df_grouped_and_dates

# weird finding where the prov:startedtime and the sosa:resultTime are the same.

# add filter to have a date range from the user , the row should be filtered by the date range
# and the sample type
example_begin_date = "2020-01-01T00:00:00Z"
example_end_date = "2021-09-01T00:00:00Z"
example_sample_type = "soil" # soil , water , hard , blank

example_filter_query = '''
PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX cvoc: <https://data.emobon.embrc.eu/ns/core_vocab#>
PREFIX prov: <http://www.w3.org/ns/prov#>
SELECT ?obs_id ?obs ?se ?start_time ?end_time ?sample_type  (COUNT(?s) AS ?observation_count)
WHERE { 
    ?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf ?se .
    ?obs core:observatoryId ?obs_id .
    ?obs core:sampleType ?sample_type .
    ?se prov:startedAtTime ?start_time .
    ?se sosa:resultTime ?end_time .
    FILTER (?start_time >= "%s"^^xsd:dateTime &&  ?end_time <= "%s"^^xsd:dateTime)
    FILTER (?sample_type = cvoc:%s)
}
GROUP BY ?obs ?sample_type ?obs_id ?se ?start_time ?end_time
''' % (example_begin_date, example_end_date, example_sample_type)

print(example_filter_query)
result_filter: QueryResult = GDB.query(sparql=example_filter_query)
df_filter: DataFrame = result_filter.to_dataframe()
df_filter



PREFIX core: <https://data.emobon.embrc.eu/ns/core#>
PREFIX smp: <https://data.emobon.embrc.eu/ns/sampling#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX cvoc: <https://data.emobon.embrc.eu/ns/core_vocab#>
PREFIX prov: <http://www.w3.org/ns/prov#>
SELECT ?obs_id ?obs ?se ?start_time ?end_time ?sample_type  (COUNT(?s) AS ?observation_count)
WHERE { 
    ?s a sosa:Observation .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf/smp:linkedToObservatory ?obs .
    ?s sosa:hasFeatureOfInterest/sosa:isResultOf ?se .
    ?obs core:observatoryId ?obs_id .
    ?obs core:sampleType ?sample_type .
    ?se prov:startedAtTime ?start_time .
    ?se sosa:resultTime ?end_time .
    FILTER (?start_time >= "2020-01-01T00:00:00Z"^^xsd:dateTime &&  ?end_time <= "2021-09-01T00:00:00Z"^^xsd:dateTime)
    FILTER (?sample_type = cvoc:soil)
}
GROUP BY ?obs ?sample_type ?obs_id ?se ?start_time ?end_time



,obs_id,obs,se,start_time,end_time,sample_type,observation_count
0,BPNS,http://data.emobon.embrc.eu/observatory-bpns-c...,http://data.emobon.embrc.eu/observatory-bpns-c...,2021-08-25,2021-08-25,https://data.emobon.embrc.eu/ns/core_vocab#soil,20
1,BPNS,http://data.emobon.embrc.eu/observatory-bpns-c...,http://data.emobon.embrc.eu/observatory-bpns-c...,2021-07-01,2021-07-01,https://data.emobon.embrc.eu/ns/core_vocab#soil,24
2,ROSKOGO,http://data.emobon.embrc.eu/observatory-roskog...,http://data.emobon.embrc.eu/observatory-roskog...,2021-08-26,2021-08-26,https://data.emobon.embrc.eu/ns/core_vocab#soil,5
3,RFormosa,http://data.emobon.embrc.eu/observatory-rformo...,http://data.emobon.embrc.eu/observatory-rformo...,2021-08-05,2021-08-05,https://data.emobon.embrc.eu/ns/core_vocab#soil,32
4,OOB,http://data.emobon.embrc.eu/observatory-oob-cr...,http://data.emobon.embrc.eu/observatory-oob-cr...,2021-08-18,2021-08-18,https://data.emobon.embrc.eu/ns/core_vocab#soil,16
5,OOB,http://data.emobon.embrc.eu/observatory-oob-cr...,http://data.emobon.embrc.eu/observatory-oob-cr...,2021-06-08,2021-06-08,https://data.emobon.embrc.eu/ns/core_vocab#soil,32
6,NRMCB,http://data.emobon.embrc.eu/observatory-nrmcb-...,http://data.emobon.embrc.eu/observatory-nrmcb-...,2021-06-21,2021-06-21,https://data.emobon.embrc.eu/ns/core_vocab#soil,20
7,NRMCB,http://data.emobon.embrc.eu/observatory-nrmcb-...,http://data.emobon.embrc.eu/observatory-nrmcb-...,2021-08-31,2021-08-31,https://data.emobon.embrc.eu/ns/core_vocab#soil,20


In [43]:
# For a named observatory (so I input the name), give me the number and percentage of missing information in the mandatory fields, per event (not per sample!). 
# NA is “missing information, by the way so don’t throw them out. Plot or tabulate this per mandatory field name (as given in the logsheets)

# first we get the list of mandatory fields from the logsheets
# this is present in a github csv file : https://raw.githubusercontent.com/emo-bon/observatory-profile/1dc9a426862798debd7791c75428630d37f9a818/logsheet_schema_extended.csv
# download the file and read it into a dataframe
url = "https://raw.githubusercontent.com/emo-bon/observatory-profile/1dc9a426862798debd7791c75428630d37f9a818/logsheet_schema_extended.csv"
df_mandatory_fields = pd.read_csv(url)
df_mandatory_fields = df_mandatory_fields[df_mandatory_fields['Requirement'] == 'mandatory']

# throw away the Notes , DataTypeIn , DataTypeOut , Format notes , Rxample, Define ourselves? , comments , LogsheetColumnDefinition away
df_mandatory_fields = df_mandatory_fields.drop(columns=['Notes', 'DataTypeIn', 'DataTypeOut', 'Format notes', 'Example', 'Define ourselves?', 'comments', 'LogsheetColumnDefinition'])

# some fields in the LogsheetType column have a ; in them , so we need to split them and explode the dataframe
# make sure to strip the whitespace from the LogsheetType column
df_mandatory_fields['LogsheetType'] = df_mandatory_fields['LogsheetType'].str.strip()
df_mandatory_fields['LogsheetType'] = df_mandatory_fields['LogsheetType'].str.split(';')
df_mandatory_fields = df_mandatory_fields.explode('LogsheetType')

# make a new df and give the count of each LogsheetType and the count of each mandatory field
df_mandatory_fields_count = df_mandatory_fields.groupby(['LogsheetType']).size().reset_index(name='count')

df_mandatory_fields
#df_mandatory_fields_count


,LogsheetColumnTitle,BaseURI,Observable_property_url,Unit,Unit_URL,LogsheetType,LogsheetTab,Requirement
49,ph,NaN,http://vocab.nerc.ac.uk/collection/S06/current...,NaN,NaN,soil,Measured,mandatory
50,ph_method,NaN,NaN,NaN,NaN,soil,Measured,mandatory
63,redox_potential,NaN,http://vocab.nerc.ac.uk/collection/P09/current...,mV,https://vocab.nerc.ac.uk/collection/P06/curren...,soil,Measured,mandatory
65,sea_subsurf_salinity,NaN,https://data.emobon.embrc.eu/ns/observableprop...,psu,http://vocab.nerc.ac.uk/collection/P06/current...,water,Measured,mandatory
66,sea_subsurf_salinity_method,NaN,NaN,NaN,NaN,water,Measured,mandatory
...,...,...,...,...,...,...,...,...
159,crate_cover,NaN,NaN,NaN,NaN,hard,Sampling,mandatory
160,size_filter,NaN,http://vocab.nerc.ac.uk/collection/P01/current...,um,http://vocab.nerc.ac.uk/collection/P06/current...,hard,Sampling,mandatory
161,images_in_PlutoF,NaN,NaN,NaN,NaN,hard,Sampling,mandatory
162,ARMS_unit_id,NaN,http://vocab.nerc.ac.uk/collection/S06/current...,NaN,NaN,hard,Observatory,mandatory
